In [1]:
# ==========================================
# PHASE 5.4 - FUTURE CLV MODEL
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

# ------------------------------------------
# Load Dataset
# ------------------------------------------

df = pd.read_csv(
    "../data/future_clv_dataset.csv"
)

print("Dataset Shape:", df.shape)

# ------------------------------------------
# Features
# ------------------------------------------

X = df[[
    "Recency",
    "Frequency",
    "PastRevenue",
    "AvgOrderValue",
    "TotalQuantity",
    "AvgQuantity",
    "ActiveDays"
]]

# ------------------------------------------
# Target
# ------------------------------------------

y = df["FutureRevenue_Log"]

# ------------------------------------------
# Train Test Split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ------------------------------------------
# Models
# ------------------------------------------

models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=10,
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            random_state=42
        )
}

# ------------------------------------------
# Train & Evaluate
# ------------------------------------------

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            preds
        )
    )

    r2 = r2_score(
        y_test,
        preds
    )

    results.append([
        name,
        mae,
        rmse,
        r2
    ])

results_df = pd.DataFrame(

    results,

    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

print("\nResults:")
print(
    results_df.sort_values(
        by="R2",
        ascending=False
    )
)

Dataset Shape: (3317, 10)

Results:
               Model       MAE      RMSE        R2
1      Random Forest  2.522773  2.912774  0.227340
2            XGBoost  2.574500  3.027463  0.165295
0  Linear Regression  2.756673  4.160048 -0.576056


In [2]:
print("FutureRevenue = 0 :", (df["FutureRevenue"] == 0).sum())
print("FutureRevenue > 0 :", (df["FutureRevenue"] > 0).sum())
print(
    "Percentage with no future purchase:",
    round((df["FutureRevenue"] == 0).mean() * 100, 2),
    "%"
)

FutureRevenue = 0 : 1365
FutureRevenue > 0 : 1952
Percentage with no future purchase: 41.15 %
